# 🐍 Master Guide: Debugging (`pdb`), Profiling (`cProfile`) & Tracing
> **Single-Source Revision Guide**: Covers interactive debugging with `breakpoint()`, CPU execution profiling (`cProfile`, `pstats`), and memory profiling (`tracemalloc`).
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Fixing complex bugs and bottlenecks requires structured diagnostics. Python includes `breakpoint()` (`pdb`), `cProfile` for CPU profiling, and `tracemalloc` for tracking memory leaks.

---
## 🧠 2. CPython Architecture: The Profiling Hook (`sys.setprofile`)

When `cProfile` runs, it installs a C-level hook into `PyEval_SetProfile()`. CPython triggers this callback on every `CALL`, `RETURN`, and `EXCEPTION` opcode, recording exact CPU cycles without instrumenting source code.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Interactive Debugging with `breakpoint()`
- `n` (next): Execute current line and advance.
- `s` (step into): Step inside called function.
- `c` (continue): Resume execution until next breakpoint.
- `p <expr>` (print): Evaluate and display variable value.

In [ ]:
def debug_demo(a, b):
    # Calling breakpoint() drops into pdb interactive debugger
    # breakpoint()
    result = a + b
    return result

print("debug_demo result:", debug_demo(10, 20))
print("Tip: Set PYTHONBREAKPOINT=0 in environment variables to disable all breakpoints in production!")

---
## 🟡 4. Part 2: Intermediate Features

### Profiling Code with `cProfile`

In [ ]:
import cProfile
import pstats

def computationally_heavy():
    return sum(x**2 for x in range(200_000))

# Profile execution:
profiler = cProfile.Profile()
profiler.enable()
computationally_heavy()
profiler.disable()

# Format and print top calls:
stats = pstats.Stats(profiler).sort_stats("tottime")
stats.print_stats(5)

### Tracking Memory Allocations with `tracemalloc`

In [ ]:
import tracemalloc

tracemalloc.start()

# Allocate objects:
data = ["allocated string block" for _ in range(50_000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Current memory: {current / 1024:.1f} KB")
print(f"Peak memory   : {peak / 1024:.1f} KB")

---
## 📋 5. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Diagnostic Tool                    | Purpose                               |
+------------------------------------+---------------------------------------+
| breakpoint()                       | Drops into pdb terminal debugger      |
| cProfile.Profile()                 | Records per-function CPU execution time|
| pstats.Stats(prof).sort_stats()    | Sorts and analyzes profile reports    |
| tracemalloc.start() / get_memory() | Tracks memory consumption & leaks     |
| PYTHONBREAKPOINT=0                 | Globally disables all breakpoints     |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Reusable Performance Profiling Decorator (`@profile_performance`)
> **Core Principle**:
> - Wrapping mission-critical routines with `cProfile` produces detailed breakdowns of total time, function call counts, and per-call averages, identifying bottlenecks instantly.

In [ ]:
# Production Function Profiling Decorator
import cProfile
import pstats
import io
import functools

def profile_performance(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        
        s = io.StringIO()
        ps = pstats.Stats(pr, stream=s).sort_stats("cumulative")
        ps.print_stats(5)  # Print top 5 time-consuming functions
        print(f"\n--- Profile for {func.__name__}() ---\n", s.getvalue())
        return result
    return wrapper

@profile_performance
def batch_student_processing():
    # Simulating data parsing and sorting
    records = [{"id": i, "score": (i * 37) % 100} for i in range(50_000)]
    sorted_records = sorted(records, key=lambda r: r["score"], reverse=True)
    return len(sorted_records)

count = batch_student_processing()
print("Processed records count:", count)

### 2. Memory Growth Diagnostics with `tracemalloc.take_snapshot()`
Tracking memory leak regressions by comparing line-by-line snapshots before and after execution.

In [ ]:
# Memory Growth Diagnostic Snapshot Drill
import tracemalloc

tracemalloc.start()
snapshot1 = tracemalloc.take_snapshot()

# Simulating memory allocation
cache_leak_simulation = ["record_data_string_" + str(i) for i in range(25_000)]

snapshot2 = tracemalloc.take_snapshot()
top_stats = snapshot2.compare_to(snapshot1, "lineno")

print("Top 3 Line-by-Line Memory Allocations:")
for stat in top_stats[:3]:
    print(" ", stat)
tracemalloc.stop()